# Assignment 9: Image Classification Using a Pre-trained Vision Transformer (ViT) and CNN

**Objective:** Perform image classification using a pre-trained Vision Transformer (ViT) and compare its performance with a pre-trained Convolutional Neural Network (CNN).

**Dataset:** CIFAR-10 (10 classes)  
**Models:** Pre-trained ViT (`google/vit-base-patch16-224-in21k`) and pre-trained ResNet-18  
**Frameworks:** PyTorch, Torchvision, Hugging Face Transformers  
**Evaluation metrics:** Test accuracy, precision, recall, F1-score, training time, and confusion matrix.

> **Execution note:** A GPU is recommended. This notebook uses a configurable subset of CIFAR-10 so it can be completed on a student machine or Google Colab. Run all cells from top to bottom. Actual results are calculated during execution; do not enter assumed accuracy values.

## 1. Introduction

Image classification is a computer vision task in which a model assigns an image to one of a set of predefined categories.

A **Convolutional Neural Network (CNN)** learns spatial patterns using convolution and pooling operations. A **Vision Transformer (ViT)** divides an image into patches, converts them into embeddings, and uses self-attention to learn relationships between patches.

In this experiment, transfer learning is used: both models start with weights learned from a large image dataset, and their classification heads are adapted to the 10 CIFAR-10 classes.

## 2. Dataset Description

CIFAR-10 contains 60,000 colour images of size 32 × 32, divided into 10 classes:
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck.

The notebook resizes images to 224 × 224 because the selected pre-trained models expect this input size. To keep runtime manageable, configurable subsets of the training and test data are used. The same subset split is used for both models.

## 3. Install and Import Libraries

In [1]:


import os
import random
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from transformers import ViTModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 4. Configure the Experiment

In [2]:
# Adjust these values depending on your computer/GPU.
TRAIN_SAMPLES = 5000
TEST_SAMPLES = 1000
BATCH_SIZE = 32
EPOCHS = 3
NUM_WORKERS = 2 if os.name != "nt" else 0

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

## 5. Load and Preprocess CIFAR-10

In [ ]:
# ImageNet normalization is used because both models were pre-trained on ImageNet.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_full = datasets.CIFAR10(root="data", train=True, download=True,
                              transform=train_transform)
test_full = datasets.CIFAR10(root="data", train=False, download=True,
                             transform=test_transform)

# Fixed, reproducible subset indices.
rng = np.random.default_rng(SEED)
train_indices = rng.permutation(len(train_full))[:min(TRAIN_SAMPLES, len(train_full))]
test_indices = rng.permutation(len(test_full))[:min(TEST_SAMPLES, len(test_full))]

train_dataset = Subset(train_full, train_indices.tolist())
test_dataset = Subset(test_full, test_indices.tolist())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"Training images: {len(train_dataset)}")
print(f"Test images: {len(test_dataset)}")

  0%|          | 360k/170M [02:00<28:58:28, 1.63kB/s]

### Display sample images

In [ ]:
def show_samples(dataset, class_names, count=8):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    for ax, idx in zip(axes.flat, range(count)):
        image, label = dataset[idx]
        image = (image * std + mean).clamp(0, 1)
        ax.imshow(image.permute(1, 2, 0))
        ax.set_title(class_names[label])
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, CLASS_NAMES)

## 6. Define the Pre-trained CNN Model (ResNet-18)

ResNet-18 is a residual CNN pre-trained on ImageNet. The original final layer is replaced with a 10-class output layer. The feature-extraction layers are frozen for a faster and more consistent transfer-learning baseline; only the new classification layer is trained.

In [ ]:
def build_resnet18(num_classes=10):
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

cnn_model = build_resnet18().to(device)
print("ResNet-18 model loaded.")

## 7. Define the Pre-trained Vision Transformer (ViT)

ViT splits the image into patches and applies transformer self-attention to model relationships between them. We use a pre-trained ViT encoder and add a linear classifier over its pooled/CLS-token representation. The encoder is frozen and the classification head is trained.

In [ ]:
class ViTClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = ViTModel.from_pretrained(
            "google/vit-base-patch16-224-in21k"
        )
        for param in self.backbone.parameters():
            param.requires_grad = False
        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        cls_embedding = outputs.last_hidden_state[:, 0]
        return self.classifier(cls_embedding)

vit_model = ViTClassifier(num_classes=len(CLASS_NAMES)).to(device)
print("ViT model loaded.")

## 8. Training and Evaluation Functions

In [ ]:
def train_model(model, train_loader, epochs=EPOCHS, model_name="model"):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=1e-3
    )
    history = {"train_loss": [], "train_accuracy": []}
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images) if model_name == "ResNet-18" else model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_accuracy = correct / total
        history["train_loss"].append(epoch_loss)
        history["train_accuracy"].append(epoch_accuracy)
        print(f"{model_name} | Epoch {epoch+1}/{epochs} "
              f"| loss={epoch_loss:.4f} | accuracy={epoch_accuracy:.4f}")

    elapsed = time.time() - start_time
    return history, elapsed


@torch.no_grad()
def evaluate_model(model, test_loader, model_name="model"):
    model.eval()
    y_true, y_pred = [], []
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1).cpu().numpy()
        y_pred.extend(predictions.tolist())
        y_true.extend(labels.numpy().tolist())

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }
    return metrics, y_true, y_pred

## 9. Train Both Models

The same training subset, test subset, number of epochs, and batch size are used for both models. Since the models and their pre-training procedures differ, this is a practical transfer-learning comparison rather than a perfectly controlled architecture-only experiment.

In [ ]:
cnn_history, cnn_train_seconds = train_model(
    cnn_model, train_loader, model_name="ResNet-18"
)

In [ ]:
vit_history, vit_train_seconds = train_model(
    vit_model, train_loader, model_name="ViT"
)

## 10. Evaluate Both Models

In [ ]:
cnn_metrics, y_true_cnn, y_pred_cnn = evaluate_model(
    cnn_model, test_loader, model_name="ResNet-18"
)
vit_metrics, y_true_vit, y_pred_vit = evaluate_model(
    vit_model, test_loader, model_name="ViT"
)

results = pd.DataFrame([cnn_metrics, vit_metrics])
results["Training time (seconds)"] = [cnn_train_seconds, vit_train_seconds]
results = results[[
    "Model", "Accuracy", "Precision (macro)", "Recall (macro)",
    "F1-score (macro)", "Training time (seconds)"
]]
display(results.round(4))

## 11. Plot Training Loss and Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS + 1), cnn_history["train_loss"], marker="o", label="ResNet-18")
axes[0].plot(range(1, EPOCHS + 1), vit_history["train_loss"], marker="o", label="ViT")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, EPOCHS + 1), cnn_history["train_accuracy"], marker="o", label="ResNet-18")
axes[1].plot(range(1, EPOCHS + 1), vit_history["train_accuracy"], marker="o", label="ViT")
axes[1].set_title("Training Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, y_true, y_pred, title in [
    (axes[0], y_true_cnn, y_pred_cnn, "ResNet-18 Confusion Matrix"),
    (axes[1], y_true_vit, y_pred_vit, "ViT Confusion Matrix")
]:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Classification Reports

In [ ]:
print("ResNet-18 classification report")
print(classification_report(y_true_cnn, y_pred_cnn, target_names=CLASS_NAMES, zero_division=0))

print("\nViT classification report")
print(classification_report(y_true_vit, y_pred_vit, target_names=CLASS_NAMES, zero_division=0))

## 14. Compare and Interpret Results

In [ ]:
# Automatically identify the model with the higher test accuracy.
best_row = results.loc[results["Accuracy"].idxmax()]
print(f"Higher test accuracy in this run: {best_row['Model']} "
      f"({best_row['Accuracy']:.4f})")

print("\nComparison table:")
display(results.sort_values("Accuracy", ascending=False).round(4))

### Discussion

Use the measured table and plots above to write your observations after running the notebook:

1. **Accuracy:** Which model achieved the higher test accuracy in this run?
2. **Precision, recall, and F1-score:** Compare the macro-averaged scores to understand performance across all classes.
3. **Training time:** Compare the time taken to train each classification head. Actual time depends on the hardware and model-loading/runtime conditions.
4. **Confusion matrix:** Identify classes that are frequently confused by each model.
5. **Limitations:** The experiment uses a subset of CIFAR-10 and frozen pre-trained feature extractors. Results may change with the sample size, epochs, hardware, and fine-tuning strategy.

Do not assume that ViT must always outperform CNN. The outcome depends on the dataset, pre-training, training setup, and computational budget.

## 15. Conclusion

This assignment demonstrates image classification using a pre-trained Vision Transformer and a pre-trained CNN on CIFAR-10. Both models are adapted to the same 10-class task using transfer learning and evaluated with accuracy, macro precision, macro recall, macro F1-score, training time, and confusion matrices.

The final conclusion should be based on the actual metrics generated by the notebook. CNNs such as ResNet-18 use convolutional filters to capture local visual patterns, while ViT uses self-attention over image patches. This comparison helps illustrate two major approaches to modern image classification.

## 16. References

1. Alex Krizhevsky, “Learning Multiple Layers of Features from Tiny Images,” 2009.
2. Kaiming He et al., “Deep Residual Learning for Image Recognition,” CVPR 2016.
3. Alexey Dosovitskiy et al., “An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale,” ICLR 2021.
4. PyTorch documentation: https://pytorch.org/docs/stable/index.html
5. Torchvision documentation: https://pytorch.org/vision/stable/index.html
6. Hugging Face Transformers documentation: https://huggingface.co/docs/transformers/index